In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("military pivot data.csv")

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("-", "_")
)

In [4]:
numeric_cols = [
    "global_rank",
    "aircraft_total_fighters",
    "armor_tanks_total",
    "navy_submarines",
    "total_population_by_country",
    "purchasing_power_parity"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[^\d.]", "", regex=True)  # removes $, tabs, spaces
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(0)

In [5]:
df["power_index_gap"] = df["global_rank"] - 1


In [6]:
total_assets = (
    df["aircraft_total_fighters"] +
    df["armor_tanks_total"] +
    df["navy_submarines"]
)

df["assets_per_capita"] = np.where(
    df["total_population_by_country"] > 0,
    total_assets / df["total_population_by_country"],
    0
)


In [7]:
df["budget_to_gdp_ratio"] = np.where(
    df["purchasing_power_parity"] > 0,
    total_assets / df["purchasing_power_parity"],
    0
)


In [8]:
asia = [
    "India","China","Japan","South Korea","North Korea","Pakistan","Bangladesh",
    "Indonesia","Iran","Iraq","Saudi Arabia","Israel","Turkey","Thailand",
    "Vietnam","Philippines","Malaysia","Singapore","Sri Lanka","Nepal"
]

europe = [
    "United Kingdom","France","Germany","Italy","Spain","Russia","Ukraine",
    "Poland","Netherlands","Belgium","Sweden","Norway","Finland","Denmark","Greece"
]

north_america = ["United States","Canada","Mexico"]
south_america = ["Brazil","Argentina","Chile","Colombia","Peru","Venezuela"]
africa = ["South Africa","Egypt","Nigeria","Kenya","Ethiopia","Morocco"]
oceania = ["Australia","New Zealand"]

def map_region(country):
    if country in asia:
        return "Asia", "Asia"
    elif country in europe:
        return "Europe", "Europe"
    elif country in north_america:
        return "North America", "North America"
    elif country in south_america:
        return "South America", "South America"
    elif country in africa:
        return "Africa", "Africa"
    elif country in oceania:
        return "Oceania", "Oceania"
    else:
        return "Other", "Other"

df[["region", "continent"]] = df["country_name"].apply(
    lambda x: pd.Series(map_region(x))
)


In [9]:
nato_members = [
    "United States","United Kingdom","France","Germany","Italy","Canada",
    "Spain","Netherlands","Belgium","Norway","Denmark","Poland"
]

df["nato_flag"] = np.where(df["country_name"].isin(nato_members), 1, 0)


In [10]:
final_df = df[
    [
        "country_name",
        "global_rank",
        "power_index_gap",
        "assets_per_capita",
        "budget_to_gdp_ratio",
        "region",
        "continent",
        "nato_flag"
    ]
]

final_df.to_excel("military_final.xlsx", index=False)

print("✅ Week-3 completed successfully. military_final.xlsx generated.")

✅ Week-3 completed successfully. military_final.xlsx generated.
